In [ ]:
#IMPORTACION DE LIBRERIAS NECESARIAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import re
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from google.colab import files

In [ ]:
#CARGUE DE DOCUMENTO
#SE CARGA LOS PERFILES DE CTD Y OBLIGATORIAMENTE LA BATIMETRIA
uploaded = files.upload()
dfs = []

In [ ]:
# Archivo A: Batimetría
df_bat = pd.read_csv('/content/Perfil_A01_A5.txt', comment='#', sep=r'\s+', header=None)
df_bat.columns = ['Distance', 'Value']

df_bat = df_bat.sort_values('Distance')
df_bat['Depth_Neg'] = df_bat['Value'] * -1

In [ ]:
# DICCIONARIO CON LAS DISTANCIAS EXACTAS para cada estación (Eje X)
mapa_distancias = {
    'A05': 1659.8,
    'A04': 2735.6,
    'A03': 3722.9,
    'A02': 4491.5,
    'A01': 5404.9
}

In [ ]:
# INTERVALO DE TEMP Y SAL PARA LA ESTACIONALIDAD
climatology = {
    'dry': {
        'temperature': {'min': 26.8, 'max': 29.9},
        'salinity': {'min': 29.2, 'max': 36.5}
    },
    'wet': {
        'temperature': {'min': 27.5, 'max': 32.5},
        'salinity': {'min': 10.0, 'max': 36.5}
    }
}

# PASO DE LAS ISOLÍNEAS (líneas negras de contorno)
paso_isohalinas = 1.0   # UPS entre isohalinas
paso_isotermas = 0.5    # °C entre isotermas

def niveles_isolineas(vmin, vmax, paso):
    """Niveles en valores redondos (múltiplos de paso) dentro de [vmin, vmax]."""
    inicio = np.ceil(vmin / paso) * paso
    return np.arange(inicio, vmax + paso * 1e-6, paso)

In [ ]:

# CREACIÓN DE LA MALLA E INTERPOLACIÓN AVANZADA - Definición de la rejilla global
# Rango de distancia y profundidad para la rejilla
dist_min, dist_max = 0, 6593.9
# prof_min se basa en el Depth_Neg mínimo del df_bat. Se actualiza para usar el df_bat filtrado.
prof_min, prof_max = df_bat['Depth_Neg'].min() - 2, 0
# Rejilla densa basada en metros (Eje X: 0 a 6593.9)
xi = np.linspace(dist_min, dist_max, 400)
yi = np.linspace(prof_min, prof_max, 400)
Xi, Yi = np.meshgrid(xi, yi)

print("Rejilla global para interpolación definida.")

In [ ]:
#PROCESO QUE ME SIMULA EL FONDO BASADO EN EL VECINO MÁS CERCANO
processed_ctd_data = []
estaciones_nombres = ['A05', 'A04', 'A03', 'A02', 'A01']

# DISTANCIA MÁXIMA DEL VECINO MÁS CERCANO
# Más allá de esta distancia horizontal (m) desde la estación más cercana,
# el campo se deja en NaN en lugar de copiar el perfil de A5 o A1 hasta el borde.
max_dist_nn = 750
dist_est = np.array([mapa_distancias[n] for n in estaciones_nombres])
dist_min_est = np.abs(Xi[..., None] - dist_est).min(axis=-1)
mascara_lejos = dist_min_est > max_dist_nn

# Límites del eje X de las figuras: exactamente el dominio interpolado
x_lim = (dist_est.min() - max_dist_nn, dist_est.max() + max_dist_nn)

# Procesamiento de los arhivos en el orden cargado
for filename, content in uploaded.items():
    if filename.endswith('.xlsx'):
        print(f"Procesando {filename}...")
        try:
            match = re.search(r'(\d{4}-\d{2}-\d{2})', filename)
            if match:
                date_str = match.group(1)
                file_date = pd.to_datetime(date_str)
            else:
                #Si el nombre del archivo no lleva la fecha, se mira adentro del archivo
                temp_df = pd.read_excel(io=content, nrows=5)
                file_date = pd.to_datetime(temp_df['Fecha[aaaa-mm-dd UT-5]'].iloc[0])

            month = file_date.month
            season = 'dry' if 1 <= month <= 4 else 'wet'
            print(f"  Fecha: {file_date.date()} | Temporada: {season}")
        except Exception as e:
            print(f"  Error al extraer fecha de {filename}: {e}")
            continue

        df_ctd_current = pd.read_excel(io=content, header=0, skiprows=[1])
        df_ctd_current = df_ctd_current.rename(columns={'Profundidad[m]': 'Pressure', 'Salinidad[UPS]': 'Salinity', 'Temperatura[degC]': 'Temperature', 'Estacion[#]': 'Station'})
        df_ctd_current['Depth_Neg'] = df_ctd_current['Pressure'] * -1
        df_ctd_current['Distance'] = df_ctd_current['Station'].astype(str).map(mapa_distancias) # Add Distance column

        #Lista de interpolación
        puntos_x = []
        puntos_y = []
        valores_salinidad = []
        valores_temperatura = []

        #Se usan los datos de ctd para la lista de interpolación
        for nombre in estaciones_nombres:
            df_sub = df_ctd_current[df_ctd_current['Station'] == nombre].dropna(subset=['Depth_Neg', 'Salinity', 'Temperature'])
            if not df_sub.empty:
                df_sub = df_sub.sort_values('Depth_Neg')
                dist_estacion = mapa_distancias[nombre]

                yi_estacion = np.linspace(df_sub['Depth_Neg'].min(), 0, 150)
                zi_sal_estacion = np.interp(yi_estacion, df_sub['Depth_Neg'], df_sub['Salinity'])
                zi_temp_estacion = np.interp(yi_estacion, df_sub['Depth_Neg'], df_sub['Temperature'])

                puntos_x.extend([dist_estacion] * len(yi_estacion))
                puntos_y.extend(yi_estacion)
                valores_salinidad.extend(zi_sal_estacion)
                valores_temperatura.extend(zi_temp_estacion)

        #Interpolación de la grilla
        Zi_salinity = griddata((puntos_x, puntos_y), valores_salinidad, (Xi, Yi), method='linear')
        mask_nan_sal = np.isnan(Zi_salinity)
        if np.any(mask_nan_sal):
            Zi_salinity_nearest = griddata((puntos_x, puntos_y), valores_salinidad, (Xi, Yi), method='nearest')
            Zi_salinity[mask_nan_sal] = Zi_salinity_nearest[mask_nan_sal]

        Zi_temperature = griddata((puntos_x, puntos_y), valores_temperatura, (Xi, Yi), method='linear')
        mask_nan_temp = np.isnan(Zi_temperature)
        if np.any(mask_nan_temp):
            Zi_temperature_nearest = griddata((puntos_x, puntos_y), valores_temperatura, (Xi, Yi), method='nearest')
            Zi_temperature[mask_nan_temp] = Zi_temperature_nearest[mask_nan_temp]

        #Aplicación del filtro gausiano
        Zi_salinity = gaussian_filter(Zi_salinity, sigma=1)
        Zi_temperature = gaussian_filter(Zi_temperature, sigma=1)

        #Límite del vecino más cercano (después del filtro: gaussian_filter propaga los NaN)
        Zi_salinity[mascara_lejos] = np.nan
        Zi_temperature[mascara_lejos] = np.nan

        #Procesamiento de datos
        processed_ctd_data.append({
            'date': file_date,
            'season': season,
            'Zi_salinity': Zi_salinity,
            'Zi_temperature': Zi_temperature
        })
        print(f"  Interpolación para {filename} completada.")

In [ ]:
#GRAFICO INDIVIDUAL PARA SAL
for data in processed_ctd_data:
    date_label = data['date'].strftime('%Y-%m-%d')
    season = data['season']
    lims = climatology[season]

    # Formateador para el eje Y sin negativos
    formatter = mticker.FuncFormatter(lambda x, p: f'{abs(x):.0f}')

    # Crear figura individual
    fig1, ax1 = plt.subplots(figsize=(14, 6), dpi=300)

    # Contorno de Salinidad
    levels_sal = np.linspace(lims['salinity']['min'], lims['salinity']['max'], 100)
    c1 = ax1.contourf(Xi, Yi, data['Zi_salinity'], levels=levels_sal, cmap='jet', extend='both')
    fig1.colorbar(c1, ax=ax1, ticks=np.linspace(lims['salinity']['min'], lims['salinity']['max'], 10))

    # Isohalinas (zorder 4.5: quedan debajo de la batimetría)
    niveles_lineas = niveles_isolineas(lims['salinity']['min'], lims['salinity']['max'], paso_isohalinas)
    lineas = ax1.contour(Xi, Yi, data['Zi_salinity'], levels=niveles_lineas, colors='black', linewidths=0.3, zorder=4.5)
    ax1.clabel(lineas, inline=True, fmt='%g', fontsize=8, colors='black')

    # Batimetría y líneas verticales
    ax1.fill_between(df_bat['Distance'], df_bat['Depth_Neg'], y2=prof_min-10, color='black', zorder=5)
    ax1.plot(df_bat['Distance'], df_bat['Depth_Neg'], color='white', lw=1.5, zorder=5)

    for n, d in mapa_distancias.items():
        ax1.axvline(d, color='white', ls='--', alpha=0.2, zorder=4)
        # Etiqueta de la estación sobre el borde superior (A05 -> A5)
        ax1.text(d, 1.01, n.replace('0', ''), transform=ax1.get_xaxis_transform(),
                 ha='center', va='bottom', fontsize=11, fontweight='bold', clip_on=False)

    # Ajustes de ejes y etiquetas
    ax1.set_xlim(*x_lim)
    ax1.set_ylim(-30, 0)
    ax1.set_ylabel('Depth')
    ax1.set_xlabel('Distance')
    ax1.yaxis.set_major_formatter(formatter)

    plt.tight_layout()
    plt.show()

    # Guardar y descargar Salinidad
    filename_sal = f"{date_label}_{season}_salinidad.png"
    fig1.savefig(filename_sal, bbox_inches='tight')
    files.download(filename_sal)
    plt.close(fig1)

In [ ]:
#GRAFICO INDIVUDIAL PARA TEM
for data in processed_ctd_data:
    date_label = data['date'].strftime('%Y-%m-%d')
    season = data['season']
    lims = climatology[season]

    # Formateador para el eje Y sin negativos
    formatter = mticker.FuncFormatter(lambda x, p: f'{abs(x):.0f}')

    # Crear figura individual
    fig2, ax2 = plt.subplots(figsize=(14, 6), dpi=300)

    # Contorno de Temperatura
    levels_temp = np.linspace(lims['temperature']['min'], lims['temperature']['max'], 100)
    c2 = ax2.contourf(Xi, Yi, data['Zi_temperature'], levels=levels_temp, cmap='turbo', extend='both')
    fig2.colorbar(c2, ax=ax2, ticks=np.linspace(lims['temperature']['min'], lims['temperature']['max'], 10))

    # Isotermas (zorder 4.5: quedan debajo de la batimetría)
    niveles_isotermas = niveles_isolineas(lims['temperature']['min'], lims['temperature']['max'], paso_isotermas)
    lineas_temp = ax2.contour(Xi, Yi, data['Zi_temperature'], levels=niveles_isotermas, colors='black', linewidths=0.3, zorder=4.5)
    ax2.clabel(lineas_temp, inline=True, fmt='%g', fontsize=8, colors='black')

    # Batimetría y líneas verticales
    ax2.fill_between(df_bat['Distance'], df_bat['Depth_Neg'], y2=prof_min-10, color='black', zorder=5)
    ax2.plot(df_bat['Distance'], df_bat['Depth_Neg'], color='white', lw=1.5, zorder=5)

    for n, d in mapa_distancias.items():
        ax2.axvline(d, color='white', ls='--', alpha=0.2, zorder=4)
        # Etiqueta de la estación sobre el borde superior (A05 -> A5)
        ax2.text(d, 1.01, n.replace('0', ''), transform=ax2.get_xaxis_transform(),
                 ha='center', va='bottom', fontsize=11, fontweight='bold', clip_on=False)

    # Ajustes de ejes y etiquetas
    ax2.set_xlim(*x_lim)
    ax2.set_ylim(-30, 0)
    ax2.set_ylabel('Depth')
    ax2.set_xlabel('Distance')
    ax2.yaxis.set_major_formatter(formatter)

    plt.tight_layout()
    plt.show()

    # Guardar y descargar Temperatura
    filename_temp = f"{date_label}_{season}_temperatura.png"
    fig2.savefig(filename_temp, bbox_inches='tight')
    files.download(filename_temp)
    plt.close(fig2)

In [ ]:
#GRAFICO MULTIPLE DE SAL#

#Estaciones
seasons_in_plots = set(data['season'] for data in processed_ctd_data)

#Calcular los valores climatológicos mínimos y máximos
colorbar_min_sal = np.inf
colorbar_max_sal = -np.inf

for season_key in seasons_in_plots:
    if climatology[season_key]['salinity']['min'] < colorbar_min_sal:
        colorbar_min_sal = climatology[season_key]['salinity']['min']
    if climatology[season_key]['salinity']['max'] > colorbar_max_sal:
        colorbar_max_sal = climatology[season_key]['salinity']['max']

# Formatter para el eje Y sin negativos
formatter = mticker.FuncFormatter(lambda x, p: f'{abs(x):.0f}')

datos_ordenados = sorted(processed_ctd_data,
                         key=lambda d: (d['date'].month, d['date'].day, d['date'].year))
num_plots = len(processed_ctd_data)

#Determinar la disposición de la cuadrícula de subgráficos
#Sujeto a archivos cargados
if num_plots == 1:
    nrows, ncols = 1, 1
    figsize = (14, 6)
elif num_plots == 2:
    nrows, ncols = 1, 2
    figsize = (14 * 2, 6)
elif num_plots <= 4:
    nrows, ncols = 2, 2
    figsize = (14 * 2, 6 * 2)
else:
    nrows = (num_plots + 1) // 2
    ncols = 2
    figsize = (14 * 2, 6 * nrows)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

for i, data in enumerate(datos_ordenados):
    ax = axes[i]
    date_label = data['date'].strftime('%Y-%m-%d')
    season = data['season']
    lims = climatology[season]

    #Contorno de Salinidad
    levels_sal_seasonal = np.linspace(lims['salinity']['min'], lims['salinity']['max'], 100)
    ax.contourf(Xi, Yi, data['Zi_salinity'], levels=levels_sal_seasonal, cmap='jet', extend='both')

    # Definimos línea negra
    niveles_lineas = niveles_isolineas(lims['salinity']['min'], lims['salinity']['max'], paso_isohalinas)
    # Dibujamos las líneas de contorno
    lineas = ax.contour(Xi, Yi, data['Zi_salinity'], levels=niveles_lineas, colors='black', linewidths=0.3, zorder=4.5)
    # Agregamos las etiquetas de texto sobre las mismas líneas
    ax.clabel(lineas, inline=True, fmt='%g', fontsize=8, colors='black')

    # Batimetría y líneas verticales
    ax.fill_between(df_bat['Distance'], df_bat['Depth_Neg'], y2=prof_min-10, color='black', zorder=5)
    ax.plot(df_bat['Distance'], df_bat['Depth_Neg'], color='white', lw=1.5, zorder=5)

    for n, d in mapa_distancias.items():
        ax.axvline(d, color='white', ls='--', alpha=0.2, zorder=4)
        # Etiqueta de la estación sobre el borde superior (A05 -> A5)
        if i < ncols:
            ax.text(d, 1.01, n.replace('0', ''), transform=ax.get_xaxis_transform(),
                    ha='center', va='bottom', fontsize=11, fontweight='bold', clip_on=False)

    # Ajustes de ejes y etiquetas
    ax.set_xlim(*x_lim)
    ax.set_ylim(-29, 0)
    if i % ncols == 0:
        ax.set_ylabel('Depth')
    else:
        ax.set_ylabel('')


    if i >= (nrows - 1) * ncols:
        ax.set_xlabel('Distance')
    else:
        ax.set_xlabel('')
    ax.yaxis.set_major_formatter(formatter)
    #ax.set_title(f'Salinity for {date_label} ({season} season)', pad=20)

#Ocultar los subgráficos no utilizados if num_plots < nrows * ncols
for j in range(num_plots, nrows * ncols):
    fig.delaxes(axes[j])

# Adjustar layout
plt.tight_layout(rect=[0, 0, 0.9, 1])

#ScalarMappable para la barra de colores utilizando la climatología estacional
sm = ScalarMappable(cmap='jet', norm=Normalize(vmin=colorbar_min_sal, vmax=colorbar_max_sal))
sm.set_array([])

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7]) # [left, bottom, width, height]
fig.colorbar(sm, cax=cbar_ax, ticks=np.linspace(colorbar_min_sal, colorbar_max_sal, 10))
cbar_ax.set_ylabel('Salinity', rotation=270, labelpad=20)

plt.show()

# Guardar y descargar
combined_filename_sal = f"combined_salinity_plots.png"
fig.savefig(combined_filename_sal, bbox_inches='tight')
files.download(combined_filename_sal)
plt.close(fig)

In [ ]:
#GRAFICO MULTIPLE DE TEMP
#Estaciones
seasons_in_plots = set(data['season'] for data in processed_ctd_data)

#Calcular los valores climatológicos mínimos y máximos
colorbar_min_temp = np.inf
colorbar_max_temp = -np.inf

for season_key in seasons_in_plots:
    if climatology[season_key]['temperature']['min'] < colorbar_min_temp:
        colorbar_min_temp = climatology[season_key]['temperature']['min']
    if climatology[season_key]['temperature']['max'] > colorbar_max_temp:
        colorbar_max_temp = climatology[season_key]['temperature']['max']

# Formatter para el eje Y sin negativos
formatter = mticker.FuncFormatter(lambda x, p: f'{abs(x):.0f}')
datos_ordenados = sorted(processed_ctd_data,
                         key=lambda d: (d['date'].month, d['date'].day, d['date'].year))
num_plots = len(processed_ctd_data)

#Determinar la disposición de la cuadrícula de subgráficos
#Sujeto a archivos cargados)
if num_plots == 1:
    nrows, ncols = 1, 1
    figsize = (14, 6)
elif num_plots == 2:
    nrows, ncols = 1, 2
    figsize = (14 * 2, 6)
elif num_plots <= 4:
    nrows, ncols = 2, 2
    figsize = (14 * 2, 6 * 2)
else:
    nrows = (num_plots + 1) // 2
    ncols = 2
    figsize = (14 * 2, 6 * nrows)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize, dpi=300, sharex=True, sharey=True)
axes = axes.flatten()

for i, data in enumerate(datos_ordenados):
    ax = axes[i]
    date_label = data['date'].strftime('%Y-%m-%d')
    season = data['season']
    lims = climatology[season]

    #Contorno de Temperatura
    levels_temp_seasonal = np.linspace(lims['temperature']['min'], lims['temperature']['max'], 100)
    ax.contourf(Xi, Yi, data['Zi_temperature'], levels=levels_temp_seasonal, cmap='turbo', extend='both')

   #Definimos líneas negras
    niveles_isotermas = niveles_isolineas(lims['temperature']['min'], lims['temperature']['max'], paso_isotermas)
     #líneas de contorno
    lineas_temp = ax.contour(Xi, Yi, data['Zi_temperature'], levels=niveles_isotermas, colors='black', linewidths=0.3, zorder=4.5)
    # Agregamos las etiquetas numéricas cortando la línea de forma limpia
    ax.clabel(lineas_temp, inline=True, fmt='%g', fontsize=8, colors='black')

    # Batimetría y líneas verticales
    ax.fill_between(df_bat['Distance'], df_bat['Depth_Neg'], y2=prof_min-10, color='black', zorder=5)
    ax.plot(df_bat['Distance'], df_bat['Depth_Neg'], color='white', lw=1.5, zorder=5)


    for n, d in mapa_distancias.items():
        ax.axvline(d, color='white', ls='--', alpha=0.2, zorder=4)
        # Etiqueta de la estación sobre el borde superior (A05 -> A5)
        if i < ncols:
            ax.text(d, 1.01, n.replace('0', ''), transform=ax.get_xaxis_transform(),
                    ha='center', va='bottom', fontsize=11, fontweight='bold', clip_on=False)

    # Ajustes de ejes y etiquetas
    ax.set_xlim(*x_lim)
    ax.set_ylim(-29, 0)
    if i % ncols == 0:
        ax.set_ylabel('Depth')
    else:
        ax.set_ylabel('')

    if i >= (nrows - 1) * ncols:
        ax.set_xlabel('Distance')
    else:
        ax.set_xlabel('')

    ax.yaxis.set_major_formatter(formatter)
    #ax.set_title(f'Temperature for {date_label} ({season} season)', pad=20)

#Ocultar los subgráficos no utilizados if num_plots < nrows * ncols
for j in range(num_plots, nrows * ncols):
    fig.delaxes(axes[j])

#Adjustar layout
plt.tight_layout(rect=[0, 0, 0.9, 1])

#ScalarMappable para la barra de colores
sm = ScalarMappable(cmap='turbo', norm=Normalize(vmin=colorbar_min_temp, vmax=colorbar_max_temp))
sm.set_array([])

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
fig.colorbar(sm, cax=cbar_ax, ticks=np.linspace(colorbar_min_temp, colorbar_max_temp, 10))
cbar_ax.set_ylabel('Temperature', rotation=270, labelpad=20)

plt.show()

#Guardar y descargar
combined_filename_temp = f"combined_temperature_plots.png"
fig.savefig(combined_filename_temp, bbox_inches='tight')
files.download(combined_filename_temp)
plt.close(fig)